In [1]:
!pip install sentence-transformers chromadb

  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 14.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 24.7 MB/s  0:00:00 eta 0:00:01
Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl (21.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

In [3]:
rag_df = pd.read_csv("../Data/rag/rico_rag_dataset.csv")

print("Shape:", rag_df.shape)
rag_df.head()

Shape: (29, 5)


,record_id,record_type,title,text,source
0,RAG_00000,component_profile,UI Component: Button,The UI component type Button appears 41 times ...,structured_component_dataset_v1_clustered.csv
1,RAG_00001,component_profile,UI Component: CheckBox,The UI component type CheckBox appears 5 times...,structured_component_dataset_v1_clustered.csv
2,RAG_00002,component_profile,UI Component: CheckedTextView,The UI component type CheckedTextView appears ...,structured_component_dataset_v1_clustered.csv
3,RAG_00003,component_profile,UI Component: EditText,The UI component type EditText appears 38 time...,structured_component_dataset_v1_clustered.csv
4,RAG_00004,cluster_profile,UI Component Cluster 0,Cluster 0 contains 128 UI components. The most...,structured_component_dataset_v1_clustered.csv


In [4]:
print(rag_df.columns.tolist())

['record_id', 'record_type', 'title', 'text', 'source']


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [6]:
texts = rag_df["text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 29
Embedding dimension: 384


In [7]:
client = chromadb.PersistentClient(path="../Data/vector_db")

print("ChromaDB initialized.")

ChromaDB initialized.


In [8]:
collection = client.get_or_create_collection(name="rico_ui_knowledge")

print("Collection created.")

Collection created.


In [9]:
collection.add(
    ids=rag_df["record_id"].tolist(),
    documents=rag_df["text"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "record_type": row["record_type"],
            "title": row["title"],
            "source": row["source"],
        }
        for _, row in rag_df.iterrows()
    ],
)

print("Documents added:", collection.count())

Documents added: 29


In [10]:
print("Documents in vector database:")
print(collection.count())

Documents in vector database:
29


In [11]:
query = "What UI components commonly appear with a Button?"

query_embedding = model.encode([query])[0]

results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)

results

{'ids': [['RAG_00000', 'RAG_00007', 'RAG_00010', 'RAG_00013', 'RAG_00019']],
 'embeddings': None,
 'documents': [['The UI component type Button appears 41 times in the analyzed RICO screenshots. Its average width is 82.84 pixels, average height is 197.30 pixels, average area is 16351.74 square pixels, and average aspect ratio is 0.42. The average detection confidence is 0.830.',
   'Button and EditText occur together in 37 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.',
   'Button and CheckBox occur together in 5 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.',
   'A Button is positioned above a EditText in the analyzed UI screenshots. This spatial relationship was observed 33 times.',
   'A Button is positioned right of a CheckBox in the analyzed UI screenshots. This spatial relationship was observed 5 times.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances

In [12]:
for i in range(len(results["documents"][0])):

    print("=" * 80)

    print("Result:", i + 1)
    print("ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])
    print("Document:")
    print(results["documents"][0][i])

    print("Metadata:")
    print(results["metadatas"][0][i])

Result: 1
ID: RAG_00000
Distance: 0.9207703471183777
Document:
The UI component type Button appears 41 times in the analyzed RICO screenshots. Its average width is 82.84 pixels, average height is 197.30 pixels, average area is 16351.74 square pixels, and average aspect ratio is 0.42. The average detection confidence is 0.830.
Metadata:
{'source': 'structured_component_dataset_v1_clustered.csv', 'title': 'UI Component: Button', 'record_type': 'component_profile'}
Result: 2
ID: RAG_00007
Distance: 1.1443208456039429
Document:
Button and EditText occur together in 37 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.
Metadata:
{'source': 'component_cooccurrence.csv', 'record_type': 'component_cooccurrence', 'title': 'Button and EditText Co-occurrence'}
Result: 3
ID: RAG_00010
Distance: 1.159500241279602
Document:
Button and CheckBox occur together in 5 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed R

In [13]:
query = "What UI elements tend to be positioned above other components?"

query_embedding = model.encode([query])[0]

results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)

for i, document in enumerate(results["documents"][0]):

    print("=" * 80)
    print("Result:", i + 1)
    print(document)

Result: 1
A Button is positioned above a EditText in the analyzed UI screenshots. This spatial relationship was observed 33 times.
Result: 2
A Button is positioned below a EditText in the analyzed UI screenshots. This spatial relationship was observed 1 times.
Result: 3
A Button is positioned right of a CheckBox in the analyzed UI screenshots. This spatial relationship was observed 5 times.
Result: 4
A Button is positioned left of a EditText in the analyzed UI screenshots. This spatial relationship was observed 3 times.
Result: 5
A CheckedTextView is positioned above a Button in the analyzed UI screenshots. This spatial relationship was observed 1 times.


In [14]:
query = "Tell me about CheckedTextView patterns"

query_embedding = model.encode([query])[0]

results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)

for i, document in enumerate(results["documents"][0]):

    print("=" * 80)
    print("Result:", i + 1)
    print(document)

Result: 1
CheckBox and CheckedTextView occur together in 3 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.
Result: 2
CheckedTextView and EditText occur together in 16 analyzed screenshots. This represents a detected co-occurrence pattern in the analyzed RICO UI data.
Result: 3
A CheckedTextView is positioned right of a CheckedTextView in the analyzed UI screenshots. This spatial relationship was observed 15 times.
Result: 4
A CheckedTextView is positioned left of a CheckedTextView in the analyzed UI screenshots. This spatial relationship was observed 12 times.
Result: 5
A CheckedTextView is positioned right of a CheckBox in the analyzed UI screenshots. This spatial relationship was observed 1 times.


In [15]:
def search_rico(query, n_results=5):

    query_embedding = model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()], n_results=n_results
    )

    for i in range(len(results["documents"][0])):

        print("=" * 80)
        print(f"Result {i + 1}")
        print(results["documents"][0][i])
        print()

In [16]:
search_rico(
    "What are the common spatial relationships between UI components?", n_results=5
)

Result 1
A Button is positioned right of a CheckBox in the analyzed UI screenshots. This spatial relationship was observed 5 times.

Result 2
A Button is positioned right of a CheckedTextView in the analyzed UI screenshots. This spatial relationship was observed 19 times.

Result 3
A CheckedTextView is positioned right of a CheckedTextView in the analyzed UI screenshots. This spatial relationship was observed 15 times.

Result 4
A CheckedTextView is positioned above a Button in the analyzed UI screenshots. This spatial relationship was observed 1 times.

Result 5
A Button is positioned left of a EditText in the analyzed UI screenshots. This spatial relationship was observed 3 times.



In [17]:
import os

vector_db_path = "../Data/vector_db"

print("Vector DB exists:", os.path.exists(vector_db_path))

print("Files:", os.listdir(vector_db_path))

Vector DB exists: True
Files: ['d3158193-ab02-46d1-9426-fdd357e6fd58', 'chroma.sqlite3']
